# Criterion Validity: Age-Split Deployment (Section 6.1)

This notebook implements the **age-split deployment** experiment from **Section 6.1** of the thesis. The proposed deployment strategy uses **MDAT for children aged 0–3** and **DEEP for children aged 3–6**, motivated by:

- MDAT's good age-tracking across the full 0–6 range.
- DEEP's design for older children (3–6) and stronger criterion validity in that age band.
- Alignment with the WHO Global Scales for Early Development (GSED).

## What this notebook does

1. Loads MDAT latent scores and filters to children aged **0–3 years**.
2. Loads DEEP latent scores and filters to children aged **3–6 years**.
3. Loads GMDS subscale scores.
4. For each age band, trains an MLP to predict GMDS from the respective instrument's latent scores.
5. Evaluates via 5-fold CV and reports per-subscale Pearson correlation.

## Inputs required

> The input file names depend on which VAE scoring run you use. Generate them by running the respective notebooks in `VAE/` and noting the output file names.

| File | Description |
|---|---|
| MDAT latent CSV (ages 0–3) | Output of `VAE/mdat_scoring.ipynb` — use rows with `Age ≤ 3×365` |
| DEEP latent CSV (ages 3–6) | Output of `VAE/DEEP_scoring.ipynb` — use rows with `Age > 3` |
| `gmds_scores.csv` | GMDS subscale raw scores |

## Relationship to thesis

Table 6.1 in the report shows the criterion validity results for this age-split approach, comparing it to using a single instrument across all ages. The age-split strategy consistently improves or matches single-instrument performance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, r2_score

target_cols = ["fol", "langCom", "eye_hand", "soc_per", "gross_mot"]

# GMDS ground truth (Age column dropped; already filtered if needed)
gmds = pd.read_csv("gmds_scores.csv")
gmds.drop(columns=["Age"], inplace=True, errors="ignore")

# MDAT band: children aged 0–3 years (Age stored in days)
mdat = pd.read_csv("mdat_Deep_abilities.csv")
mdat = mdat[mdat["Age"] <= 3 * 365][["child_ids", "tot"]].drop_duplicates(subset=["child_ids"])

# DEEP band: children aged 3–6 years (Age stored in years)
deep = pd.read_csv("DEEP_Deep_abilities.csv")
deep = deep[deep["Age"] > 3][["child_ids", "total_DEEP"]].drop_duplicates(subset=["child_ids"])

# Merge each age band with GMDS targets
df_mdat = pd.merge(mdat, gmds, on="child_ids").dropna()
df_deep = pd.merge(deep, gmds, on="child_ids").dropna()

print(f"MDAT band (0–3 yrs): {len(df_mdat)} children with GMDS")
print(f"DEEP band (3–6 yrs): {len(df_deep)} children with GMDS")

## Evaluation Function

The same 5-fold CV MLP used in `base_model_single_NN_for_criterion.ipynb` (see that notebook for full model justification). Evaluated separately for each age band.

In [ ]:
def evaluate_model_per_target(model, X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_corrs, fold_mses, fold_r2s = [], [], []

    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model_clone = clone(model)
        model_clone.fit(X_train, y_train)
        y_pred = model_clone.predict(X_test)

        y_pred = np.atleast_2d(y_pred)
        if y_pred.shape[0] != len(X_test):
            y_pred = y_pred.T

        y_true = y_test.values
        y_true = np.atleast_2d(y_true)
        if y_true.shape[0] != len(X_test):
            y_true = y_true.T

        corrs, mses, r2s = [], [], []
        for i in range(y_true.shape[1]):
            corrs.append(pd.Series(y_pred[:, i]).corr(pd.Series(y_true[:, i])))
            mses.append(mean_squared_error(y_true[:, i], y_pred[:, i]))
            r2s.append(r2_score(y_true[:, i], y_pred[:, i]))

        fold_corrs.append(corrs)
        fold_mses.append(mses)
        fold_r2s.append(r2s)

    fold_corrs = np.array(fold_corrs)
    fold_mses  = np.array(fold_mses)
    fold_r2s   = np.array(fold_r2s)

    return pd.DataFrame({
        "Target":    y.columns,
        "Mean Corr": np.round(fold_corrs.mean(axis=0), 2),
        "Std Corr":  np.round(fold_corrs.std(axis=0), 3),
        "Mean MSE":  fold_mses.mean(axis=0),
        "Mean R2":   fold_r2s.mean(axis=0),
    })


mlp_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("model",   MLPRegressor(
        hidden_layer_sizes=(32,),
        alpha=1e-4,
        max_iter=2000,
        random_state=42,
    )),
])

## MDAT Band: Ages 0–3 Years

Predict GMDS subscales using only the MDAT total latent score for children aged ≤ 3 years.

In [ ]:
X_mdat = df_mdat[["tot"]]
y_mdat = df_mdat[target_cols]

mdat_results = evaluate_model_per_target(mlp_model, X_mdat, y_mdat)

print(f"MDAT band — n={len(df_mdat)}")
display(mdat_results)

## DEEP Band: Ages 3–6 Years

Predict GMDS subscales using only the DEEP total latent score for children aged > 3 years.

In [ ]:
X_deep = df_deep[["total_DEEP"]]
y_deep = df_deep[target_cols]

deep_results = evaluate_model_per_target(mlp_model, X_deep, y_deep)

print(f"DEEP band — n={len(df_deep)}")
display(deep_results)

## Combined Summary (Table 6.1)

Side-by-side comparison of the two age bands. These numbers populate Table 6.1 in the thesis.

In [ ]:
summary = mdat_results[["Target", "Mean Corr", "Std Corr"]].rename(
    columns={"Mean Corr": "MDAT r (0–3 yrs)", "Std Corr": "MDAT std"}
).merge(
    deep_results[["Target", "Mean Corr", "Std Corr"]].rename(
        columns={"Mean Corr": "DEEP r (3–6 yrs)", "Std Corr": "DEEP std"}
    ),
    on="Target",
)

print("Table 6.1 — Age-Split Criterion Validity (mean Pearson r, 5-fold CV)")
display(summary)